# Capacity Ladder — is it the signal, or the network's size?

**The question.** Our model scores ~0.847 with the tree bits and ~0.815 without them. Is that
lift coming from the **information in the bits**, or from the network being **big enough to
exploit anything**? And since the last experiment concluded that *the memorisation lives in the
network, not in the bits*, this is the direct test of that claim: shrink the network and see
whether the memorisation finally stops.

**The method.** Shrink the TabResNet trunk over four levels — a **56× parameter reduction** —
and run **both views at every level**. The x-only arm is essential: it shows whether a small
model can still learn at all, so a drop in the x+tree arm can be attributed to capacity rather
than to the shrink breaking training.

| arm | d | d_hidden | blocks | params (x+tree) | params (x only) | shrink |
|---|---|---|---|---|---|---|
| **full** (current) | 256 | 512 | 4 | 2,439,682 | 1,057,538 | 1× |
| medium | 64 | 128 | 2 | 379,906 | 34,370 | 6.4× |
| small | 16 | 32 | 1 | 87,730 | 1,346 | 27.8× |
| **tiny** | 8 | 16 | 0 | 43,314 | 122 | 56.3× |

The tiny arm has **no residual blocks at all** — it is a linear map of the bits into 8
dimensions and then out to the two classes. If that still scores ~0.847, the tree bits are
carrying the result almost by themselves.

**Everything else is frozen** and identical to the last three experiments: credit, OOB-honest
hard bits, AdamW (lr 3e-4, batch 128), 400 epochs with best-validation selection, 2-member
ensembles, dropout and L1 off. Learning rate is deliberately **not** retuned per arm, so the
only variable is size.

**What to look for**
1. **x+tree stays flat as the model shrinks** → the lift is the signal, not the capacity.
2. **The gap between the two curves widens as capacity falls** → the bits are doing work the
   raw features cannot do at that size.
3. **ep99** (the epoch train AUC first reaches 0.99) **moves later in the small arms** → capacity
   really is the memorisation channel, confirming the last report's conclusion.

⏱ **~50 minutes on an A100** (the small arms are fast). Runtime → GPU → Run all.

**Per-batch logging is on, for every batch of every epoch.** Each mini-batch prints its loss, the running-average loss within the epoch, its accuracy and its AUC, under the epoch it belongs to. That is 88 batches per epoch (11,200 train rows / 128) x 400 epochs, so expect **~35,000 lines per model and ~560,000 in total**. Colab's tab can get sluggish at that volume, so every run is also teed to a log file under `results/fusion/logs/` and every batch is written to `fusion_credit_batches.csv` — the full record survives regardless of what the browser does. If you ever want it lighter: `--log-batch-every 10` (print every 10th batch) or `--log-batch-epochs 20` (first 20 epochs only).

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 3b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import os, glob, tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 4 · shared recipe — identical to the last three experiments; ONLY the size changes below
#     --log-batches prints EVERY batch of EVERY epoch (88 batches x 400 epochs per model)
import os
os.makedirs('results/fusion/logs', exist_ok=True)
base = ('--task 361055 --views "x;x+tree" --encoding oob --ensemble 2 --epochs 400 '
        '--dropout 0 --l1 0 --weight-decay 1e-3 --lr 3e-4 --batch-size 128 --device auto '
        '--log-batches')
print(base)
print('\n88 batches per epoch -> ~35,000 batch lines per model, ~560,000 in total.')
print('Each run is also teed to results/fusion/logs/<arm>.log')

In [ ]:
# 5 · THE CAPACITY LADDER — each run trains x-only AND x+tree at that size
#     every batch is printed AND teed to a log file (the tab may lag; the files won't)
!python -u run_fusion.py {base} --d 256 --d-hidden 512 --n-blocks 4  --out results/fusion/cap_1_full   2>&1 | tee results/fusion/logs/cap_1_full.log
!python -u run_fusion.py {base} --d  64 --d-hidden 128 --n-blocks 2  --out results/fusion/cap_2_medium 2>&1 | tee results/fusion/logs/cap_2_medium.log
!python -u run_fusion.py {base} --d  16 --d-hidden  32 --n-blocks 1  --out results/fusion/cap_3_small  2>&1 | tee results/fusion/logs/cap_3_small.log
!python -u run_fusion.py {base} --d   8 --d-hidden  16 --n-blocks 0  --out results/fusion/cap_4_tiny   2>&1 | tee results/fusion/logs/cap_4_tiny.log

In [ ]:
# 6 · summary table
import json, glob, pandas as pd
rows = []
for d in sorted(glob.glob('results/fusion/cap_*')):
    js = glob.glob(d + '/fusion_*.json')
    if not js: continue
    s = json.load(open(js[0]))
    e = pd.read_csv(glob.glob(d + '/fusion_*_epochs.csv')[0])
    for r in s['results']:
        sub = e[e.model == r['model']].groupby('epoch').train_auc.mean()
        hit = sub[sub >= 0.99]
        rows.append(dict(arm=d.split('cap_')[-1], views=r['views'],
                         d=r['d'], blocks=r['n_blocks'], params=r['params'],
                         test_auc=r['test_auc'], best_val=r['best_val_auc'],
                         best_ep=r['best_epoch'],
                         final_train=round(sub.iloc[-1], 4),
                         ep99=int(hit.index.min()) if len(hit) else None,
                         ceiling=s['tree_ceiling']))
t = pd.DataFrame(rows)
print(t.round(4).to_string(index=False))
print('\n--- the tree-bit lift at each size ---')
for arm in t.arm.unique():
    a = t[t.arm == arm]
    xo = a[a.views == 'x'].test_auc
    xt = a[a.views == 'x+tree'].test_auc
    if len(xo) and len(xt):
        print(f'  {arm:10s} params={int(a[a.views=="x+tree"].params.iloc[0]):>9,d}  '
              f'x={xo.iloc[0]:.4f}  x+tree={xt.iloc[0]:.4f}  lift={xt.iloc[0]-xo.iloc[0]:+.4f}')
t.to_csv('results/fusion/cap_summary.csv', index=False)

In [ ]:
# 7 · the capacity curve + the memorisation gauge
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
a = ax[0]
ceil = t.ceiling.iloc[0]
for views, col, lab in [('x+tree', '#2a78d6', 'x + tree bits'), ('x', '#eb6834', 'x only (no bits)')]:
    s_ = t[t.views == views].sort_values('params')
    a.plot(s_.params, s_.test_auc, 'o-', color=col, ms=8, lw=2, label=lab, mec='white', mew=1.2)
a.axhline(ceil, ls='--', color='#52514e', lw=1.2, label=f'best tree {ceil:.4f}')
a.set_xscale('log'); a.set_xlabel('trainable parameters (log scale)'); a.set_ylabel('test AUC')
a.set_title('Does the lift survive a 56x smaller network?', fontweight='bold', loc='left')
a.legend(fontsize=9, frameon=False); a.grid(alpha=.25)
b = ax[1]
s_ = t[t.views == 'x+tree'].sort_values('params')
lab = [f"{r.arm}\n{int(r.params):,}" for _, r in s_.iterrows()]
vals = [r.ep99 if r.ep99 else 400 for _, r in s_.iterrows()]
b.bar(range(len(s_)), vals, color='#2a78d6', width=.6, edgecolor='white', lw=1.5)
for i, v in enumerate(vals): b.text(i, v + 0.3, str(v), ha='center', fontsize=10)
b.set_xticks(range(len(s_))); b.set_xticklabels(lab, fontsize=8.5)
b.set_ylabel('epoch train AUC first reaches 0.99'); b.grid(alpha=.25, axis='y')
b.set_title('Does a smaller network stop memorising?', fontweight='bold', loc='left')
plt.tight_layout(); plt.savefig('results/fusion/cap_overview.png', dpi=150); plt.show()

In [ ]:
# 7b · batch-by-batch curves — full vs tiny (all 400 epochs of batches are in the CSV)
import pandas as pd, glob, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for axx, arm, ttl in [(ax[0], 'cap_1_full', 'full — 2.44M params'),
                      (ax[1], 'cap_4_tiny', 'tiny — 43k params')]:
    f = glob.glob(f'results/fusion/{arm}/fusion_*_batches.csv')
    if not f: continue
    b = pd.read_csv(f[0])
    b = b[(b.model == 'x + tree') & (b.member == 0)]
    axx.plot(b.step, b.batch_loss, lw=.7, color='#c8c8c4', label='single batch')
    axx.plot(b.step, b.batch_loss.rolling(44, min_periods=1).mean(), lw=2,
             color='#2a78d6', label='half-epoch rolling mean')
    nb_ = int(b.batch.max())
    for e in range(1, int(b.epoch.max()) + 1):
        axx.axvline(e * nb_, color='#eb6834', lw=.4, alpha=.5)
    axx.set_title(f'{ttl}  (orange lines = epoch boundaries)', fontweight='bold',
                  loc='left', fontsize=10.5)
    axx.set_xlabel('mini-batch step'); axx.grid(alpha=.25)
    axx.legend(fontsize=9, frameon=False)
ax[0].set_ylabel('training loss on the batch')
plt.tight_layout(); plt.savefig('results/fusion/cap_batches.png', dpi=150); plt.show()

# how fast the loss falls inside the very first epoch
for arm in ['cap_1_full', 'cap_4_tiny']:
    f = glob.glob(f'results/fusion/{arm}/fusion_*_batches.csv')
    if not f: continue
    b = pd.read_csv(f[0])
    b = b[(b.model == 'x + tree') & (b.member == 0) & (b.epoch == 1)]
    print(f'{arm:12s} epoch 1: loss {b.batch_loss.iloc[0]:.4f} -> {b.batch_loss.iloc[-1]:.4f}, '
          f'batch AUC {b.batch_auc.iloc[0]:.3f} -> {b.batch_auc.iloc[-1]:.3f}')

In [ ]:
# 8 · download everything (results, per-batch CSVs, figures and the full logs)
import shutil, glob, os
for f in glob.glob('results/fusion/*/fusion_*_batches.csv'):
    print(f'{os.path.getsize(f)/1e6:7.1f} MB  {f}')
for f in glob.glob('results/fusion/logs/*.log'):
    print(f'{os.path.getsize(f)/1e6:7.1f} MB  {f}')
shutil.make_archive('capacity_ladder_credit', 'zip', 'results/fusion')
print(f"\nzip: {os.path.getsize('capacity_ladder_credit.zip')/1e6:.1f} MB")
from google.colab import files
files.download('capacity_ladder_credit.zip')